## Transform Circuits Data

In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/3.silver_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

#### Step 1 - Reading bronze `circuits` table


In [0]:
from pyspark.sql import functions as F
circuits_df = (
    spark.table(bronze_table)
    .filter(F.col("batch_id") == v_batch_id)
)


In [0]:
display(circuits_df)

#### Step 2 - Dropping url column


In [0]:
from pyspark.sql import functions as F

circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),  
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

In [0]:
display(circuits_selected_df)

#### Step 3 & 4 - Standardising Column Names
 

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

In [0]:
display(circuits_renamed_df)

#### Step 5 - Filtering out rows where circuit_id is null (business key validation)

In [0]:
circuits_filtered_df = (
    circuits_renamed_df
    .filter(
        F.col("circuit_id").isNotNull()
    )
)

In [0]:
display(circuits_filtered_df)

#### Step 6 - Removing duplicate records

In [0]:
circuits_distinct_df = (
    circuits_filtered_df
    .dropDuplicates(["circuit_id"])
)

In [0]:
display(circuits_distinct_df)   

#### Step 7 - Transforming values of columns `circuit_name` and `locality` to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df.withColumns(
        {
            "circuit_name": F.initcap(F.col("circuit_name")),
            "locality": F.initcap(F.col("locality"))
        }
    )
)

In [0]:
display(circuits_final_df)

### Step 8 - Write the transformed data to silver `circuits` table


In [0]:
circuit_columns_to_update = [
    "circuit_id",
    "circuit_name",
    "latitude",
    "longitude",
    "locality",
    "country",
    "ingestion_timestamp",
    "source_file",
    "batch_id",
]

write_to_silver(
    circuits_final_df, 
    silver_table,
    merge_condition = "t.circuit_id = s.circuit_id",
    columns_to_update = circuit_columns_to_update
    )

In [0]:
display(spark.table(silver_table))